# 11 — GEB Paper Empirik Güçlendirme DeneyleriBu notebook GEB makalesi (v0.8) için **dört empirik güçlendirme deneyini** koşturur. Her deney bağımsız, başında bir `RUN_*` bayrağı ile aç/kapa.## Deneyler| Deney | Amaç | Süre (A100) | Ne gerek? ||---|---|---|---|| **A. NBS k≥3 robustness** | Section 4.3'teki sampling bias kaygısını test eder; 6 koalisyon için filtered vs unfiltered NBS karşılaştırması | 5-10 dk | Checkpoint || **B. Shapley CI'lar** | 10-seed bootstrap ile her üye Shapley'i için 95% CI; Section 5 tablolarına ±SE eklemek için | 15-20 dk | Checkpoint || **C. Multi-task ablation** | λ_dur=λ_coh=0 retrain; Section 3.5/6.2'deki ablation TBD'lerini gerçek sayılarla doldurur | 20-30 dk | Tam veri + training || **D. Multi-seed retrain** | 5 seed retrain; Section 5 metriclerine seed-aralığı | 60-90 dk | Tam veri + training |## Çıktı formatıHer deney sonunda **paylaşılabilir paragraf özeti** üretilir, Claude'a yapıştırılabilir.## Notlar- A ve B sadece checkpoint kullanır → veri yoksa bile koşar- C ve D tam training pipeline gerektirir (snapshots, coalitions, country_master)- Tüm rastgelelik seed'lerle kontrollü

## Yapılandırma — hangi deneyleri koşalım?

In [ ]:
# Hangi deneyler koşulacak? (True/False bayrakları)
RUN_A_NBS_ROBUSTNESS = True   # 5-10 dk
RUN_B_SHAPLEY_CI     = True   # 15-20 dk
RUN_C_MULTITASK_ABL  = True  # 20-30 dk, full data gerek
RUN_D_MULTISEED      = True  # 60-90 dk, full data gerek

# Multi-seed sayısı (D için)
MULTISEED_N = 5
MULTISEED_SEEDS = [42, 137, 256, 511, 1024]

# Bootstrap iterations (B için)
B_BOOTSTRAP_SEEDS = list(range(42, 52))  # 10 seed

# Drive paths
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/coalition_gnn'
SNAP_DIR    = os.path.join(PROJECT_DIR, 'data/snapshots')
COAL_DIR    = os.path.join(PROJECT_DIR, 'data/coalitions')
MODELS_DIR  = os.path.join(PROJECT_DIR, 'models')
CANON_DIR   = os.path.join(PROJECT_DIR, 'data/canonical')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'geb_experiments')
os.makedirs(RESULTS_DIR, exist_ok=True)

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'Deneyler aktif: A={RUN_A_NBS_ROBUSTNESS}, B={RUN_B_SHAPLEY_CI}, C={RUN_C_MULTITASK_ABL}, D={RUN_D_MULTISEED}')

Mounted at /content/drive
Device: cuda
Deneyler aktif: A=True, B=True, C=True, D=True


In [ ]:
!pip install -q torch torch_geometric numpy pandas tqdm scikit-learn matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 79.4 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import RGCNConv
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from tqdm.auto import tqdm
from itertools import combinations
from scipy.optimize import linprog
import random
import time
import json as json_mod

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

EDGE_TYPES = ['allies', 'trades', 'votes_with', 'conflicts_with']
NUM_RELATIONS = len(EDGE_TYPES)

START_YEAR, END_YEAR = 1946, 2016
TRAIN_END = 1999
VAL_END   = 2009

## Model mimarisi (07 ile birebir aynı)

In [ ]:
class RGCNEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_relations, dropout=0.3):
        super().__init__()
        self.conv1 = RGCNConv(in_channels, hidden_channels, num_relations)
        self.conv2 = RGCNConv(hidden_channels, out_channels, num_relations)
        self.dropout = dropout
        self.norm1 = nn.LayerNorm(hidden_channels)
        self.norm2 = nn.LayerNorm(out_channels)
    def forward(self, x, ei, et):
        h = self.conv1(x, ei, et); h = self.norm1(h); h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = self.conv2(h, ei, et); h = self.norm2(h)
        return h

class RichCoalitionHead(nn.Module):
    def __init__(self, embed_dim, raw_dim, hidden_dim=128, dropout=0.3):
        super().__init__()
        pool_dim = 4 * (embed_dim + raw_dim) + 2
        self.W_pair = nn.Parameter(torch.randn(embed_dim, embed_dim) * 0.01)
        self.validity_mlp = nn.Sequential(
            nn.Linear(pool_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1))
        embed_pool_dim = 4 * embed_dim + 1
        self.duration_head = nn.Sequential(
            nn.Linear(embed_pool_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 1), nn.Softplus())
        self.cohesion_head = nn.Sequential(
            nn.Linear(embed_pool_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 1), nn.Sigmoid())
    def rich_pool(self, f):
        std = f.std(dim=0) if f.shape[0] > 1 else torch.zeros_like(f[0])
        return torch.cat([f.mean(dim=0), std, f.max(dim=0).values, f.min(dim=0).values])
    def forward(self, z, x_raw):
        k = z.shape[0]
        size_feat = torch.tensor([k/20.0], device=z.device)
        z_pool = self.rich_pool(z); raw_pool = self.rich_pool(x_raw)
        if k >= 2:
            Wz = z @ self.W_pair; pm = z @ Wz.T
            pair = (pm.sum() - pm.diag().sum()) / (k*(k-1))
        else:
            pair = torch.tensor(0.0, device=z.device)
        v = self.validity_mlp(torch.cat([z_pool, raw_pool, size_feat, pair.unsqueeze(0)])).squeeze()
        mt = torch.cat([z_pool, size_feat])
        return v, self.duration_head(mt).squeeze(), self.cohesion_head(mt).squeeze()

class FullRichGNN(nn.Module):
    def __init__(self, num_features, hidden_dim=128, embed_dim=128, num_relations=4, head_hidden=128, dropout=0.3):
        super().__init__()
        self.encoder = RGCNEncoder(num_features, hidden_dim, embed_dim, num_relations, dropout)
        self.head = RichCoalitionHead(embed_dim, num_features, head_hidden, dropout)
    def forward_sample(self, snap, member_idx_list):
        z_all = self.encoder(snap['x'], snap['edge_index'], snap['edge_type'])
        idx = torch.tensor(member_idx_list, dtype=torch.long, device=DEVICE)
        return self.head(z_all[idx], snap['x'][idx])

print('Model sınıfları yüklendi')

Model sınıfları yüklendi


## Veri yükleme (snapshots + coalitions)

In [ ]:
def load_snapshot(year):
    path = os.path.join(SNAP_DIR, f'graph_{year}.pt')
    if not os.path.exists(path):
        return None
    data = torch.load(path, weights_only=False)
    x = data['country'].x
    cow_codes = data['country'].cow_code.numpy()

    edge_index_list, edge_type_list = [], []
    for rel_idx, rel_name in enumerate(EDGE_TYPES):
        key = ('country', rel_name, 'country')
        if key in data.edge_types:
            ei = data[key].edge_index
            edge_index_list.append(ei)
            edge_type_list.append(torch.full((ei.shape[1],), rel_idx, dtype=torch.long))

    if not edge_index_list:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_type  = torch.zeros(0, dtype=torch.long)
    else:
        edge_index = torch.cat(edge_index_list, dim=1)
        edge_type  = torch.cat(edge_type_list)

    return {
        'x': x.to(DEVICE),
        'edge_index': edge_index.to(DEVICE),
        'edge_type': edge_type.to(DEVICE),
        'cow_codes': cow_codes,
        'code_to_idx': {int(c): i for i, c in enumerate(cow_codes)},
        'year': year,
    }

snapshots = {}
for year in range(START_YEAR, END_YEAR + 1):
    snap = load_snapshot(year)
    if snap is not None:
        snapshots[year] = snap

NUM_FEATURES = next(iter(snapshots.values()))['x'].shape[1]
print(f'Snapshots: {len(snapshots)} yıl, {NUM_FEATURES} özellik')

# Country name mapping
country_master = pd.read_parquet(os.path.join(CANON_DIR, 'country_master.parquet'))
cow_to_name = {int(row['cow_code']): row.get('state_name', f"COW{row['cow_code']}") for _, row in country_master.iterrows()}
def names(cow_list): return [cow_to_name.get(c, str(c)) for c in cow_list]

Snapshots: 71 yıl, 10 özellik


## Koalisyon listeleri (paper'daki 6 koalisyon-yıl)

In [ ]:
# COW codes
warsaw_members = [265, 290, 310, 315, 339, 355, 360, 365]
# GDR(265), POL(290), HUN(310), CZE(315), ALB(339), BUL(355), ROM(360), USSR(365)

nato_26 = [2,20,200,210,211,212,220,230,235,255,290,310,316,317,325,
           349,350,355,360,366,367,368,385,390,395,640]

eu_27 = [200,205,210,211,212,220,230,235,255,290,305,310,316,317,325,
         338,349,350,352,355,360,366,367,368,375,380,390]

asean_10 = [775,800,811,812,816,820,830,835,840,850]

brics = [140, 365, 560, 710, 750]

coalitions_paper = [
    ('Warsaw Pact 1980',  warsaw_members, 1980),
    ('Warsaw Pact 1989',  warsaw_members, 1989),
    ('BRICS 2014',        brics,           2014),
    ('ASEAN-10 2010',     asean_10,        2010),
    ('NATO-26 2010',      nato_26,         2010),
    ('EU-27 2010',        eu_27,           2010),
]
print(f'{len(coalitions_paper)} koalisyon-yıl tanımlandı')

6 koalisyon-yıl tanımlandı


## v(S) hesaplama fonksiyonu (model üzerinde)

In [ ]:
def compute_v(model, members_cow, year, return_logit=True):
    """v(S) skoru — varsayılan logit (Section 3.2'ye uygun)."""
    if year not in snapshots:
        return None
    snap = snapshots[year]
    valid = [m for m in members_cow if m in snap['code_to_idx']]
    if len(valid) < 2:
        return None
    idx = [snap['code_to_idx'][m] for m in valid]
    with torch.no_grad():
        logit, _, _ = model.forward_sample(snap, idx)
    if return_logit:
        return logit.item()
    return torch.sigmoid(logit).item()

## Checkpoint'i yükle (A ve B için ana model)

In [ ]:
ckpt_path = os.path.join(MODELS_DIR, 'full_rich_gnn.pt')
model = FullRichGNN(NUM_FEATURES).to(DEVICE)
model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=False))
model.eval()
print(f'✓ Checkpoint yüklendi: {ckpt_path}')

# Sanity check
test_v = compute_v(model, warsaw_members, 1980, return_logit=True)
print(f'Sanity: Warsaw 1980 v(N) logit = {test_v:+.4f}  (paper: +3.05)')

✓ Checkpoint yüklendi: /content/drive/MyDrive/coalition_gnn/models/full_rich_gnn.pt
Sanity: Warsaw 1980 v(N) logit = +3.0508  (paper: +3.05)


---# Deney A: NBS k≥3 Robustness Test**Amaç:** Section 4.3'te kabul edilen sampling bias kaygısını test et.**Problem:** Marginal-threat NBS sampling'inde, rastgele çekilen k=1 olduğunda, S\{i} tek elemanlı kalıyor ve `compute_v` `None` döndürüyor; bu durumda `v_without=0` alınıyor. Bu durum min statistic'i şişirebilir.**Çözüm:** k≥3 filtresi ile NBS'i yeniden hesapla, unfiltered ile karşılaştır.**Beklenti:** Hungary 1989 NBS+0.62 sonucu paper'da kritik. Bu değer sağlam mı?

In [ ]:
def compute_nbs_with_filter(model, members, year, n_samples=50, min_k=3, seed=42, use_logit=True):
    """NBS with optional k-filter. min_k=1 unfiltered (eski davranış), min_k=3 filtered."""
    rng = np.random.default_rng(seed)
    v_N = compute_v(model, members, year, return_logit=use_logit)
    if v_N is None: return None

    threats = {}
    n = len(members)
    for m in members:
        other = [x for x in members if x != m]
        min_marg = float('inf')
        n_valid = 0
        for _ in range(n_samples):
            # k çekme aralığı: min_k-1 ile n-1 arası
            # (k = S büyüklüğü, S\{m} büyüklüğü = k-1 = en az min_k-1 olmalı)
            k = rng.integers(max(min_k, 2), n)  # min_k → S\{m} en az min_k-1 elemanlı
            sub = list(rng.choice(other, min(k, len(other)), replace=False))
            S_with    = sub + [m]
            S_without = sub
            if len(S_without) < 2:
                continue  # filter: skip if v(S\{i}) undefined
            v_with    = compute_v(model, S_with,    year, return_logit=use_logit)
            v_without = compute_v(model, S_without, year, return_logit=use_logit)
            if v_with is not None and v_without is not None:
                marg = v_with - v_without
                if marg < min_marg:
                    min_marg = marg
                n_valid += 1
        threats[m] = (min_marg if min_marg != float('inf') else 0.0, n_valid)

    total_threat = sum(t for t, _ in threats.values())
    surplus = v_N - total_threat
    allocation = {m: threats[m][0] + surplus / n for m in members}
    return {
        'allocation': allocation,
        'threats': {m: threats[m][0] for m in members},
        'n_valid_samples': {m: threats[m][1] for m in members},
        'v_N': v_N,
    }

if RUN_A_NBS_ROBUSTNESS:
    print('=' * 72)
    print('DENEY A — NBS k≥3 Robustness Test')
    print('=' * 72)

    results_A = {}
    for name, members, year in coalitions_paper:
        print(f'\n--- {name} ---')

        # Unfiltered (eski davranış, min_k=2 → S\{m} = 1 elemanlı olabilir → v_without=0)
        # Yukarıdaki kod min_k=2 ile bile filter yapıyor (continue if <2),
        # bu yüzden "unfiltered eski"yi taklit için min_k=2 ile koşalım + filter'sız version
        # Doğru karşılaştırma: min_k=2 (en sıkı filter) vs min_k=3
        # Veya: skip-filter (orijinal kod, v_without=0 fallback) vs k≥3
        # Burada "k>=2 → S\{m}>=1, v(S\{m})=0 if <2" eski davranışı manuel oluşturalım

        # Orijinal davranış: min_k=1 olabilir, v_without=0 fallback (sampling bias var)
        def nbs_original(seed_):
            rng = np.random.default_rng(seed_)
            v_N = compute_v(model, members, year, return_logit=True)
            threats = {}
            n = len(members)
            for m in members:
                other = [x for x in members if x != m]
                min_marg = float('inf')
                for _ in range(100):
                    k = rng.integers(3, n)  # k>=1
                    sub = list(rng.choice(other, min(k, len(other)), replace=False))
                    S_with = sub + [m]
                    S_without = sub
                    v_with = compute_v(model, S_with, year, return_logit=True)
                    v_without = compute_v(model, S_without, year, return_logit=True) if len(S_without) >= 2 else 0.0  # BIAS!
                    if v_with is not None and v_without is not None:
                        marg = v_with - v_without
                        if marg < min_marg:
                            min_marg = marg
                threats[m] = min_marg if min_marg != float('inf') else 0.0
            total = sum(threats.values())
            surplus = v_N - total
            return {m: threats[m] + surplus / n for m in members}, threats, v_N

        orig_alloc, orig_threats, v_N = nbs_original(42)

        # Filtered: k>=3 → S\{m} >= 2 (her zaman valid)
        filtered = compute_nbs_with_filter(model, members, year, n_samples=300, min_k=3, seed=42)

        # Karşılaştırma tablosu
        print(f'{"Üye":<12} {"Orig NBS":>10} {"Filt NBS":>10} {"Δ":>8}')
        print('-' * 44)
        for m in members:
            o = orig_alloc[m]
            f = filtered['allocation'][m]
            delta = f - o
            mname = cow_to_name.get(m, str(m))[:11]
            print(f'{mname:<12} {o:>+10.4f} {f:>+10.4f} {delta:>+8.4f}')

        results_A[name] = {
            'members': members,
            'year': year,
            'v_N': v_N,
            'orig_allocation': {str(m): float(orig_alloc[m]) for m in members},
            'orig_threats':    {str(m): float(orig_threats[m]) for m in members},
            'filtered_allocation': {str(m): float(filtered['allocation'][m]) for m in members},
            'filtered_threats':    {str(m): float(filtered['threats'][m]) for m in members},
            'max_abs_delta': max(abs(filtered['allocation'][m] - orig_alloc[m]) for m in members),
        }

    # Sonuçları kaydet
    with open(os.path.join(RESULTS_DIR, 'A_nbs_robustness.json'), 'w') as f:
        json_mod.dump(results_A, f, indent=2)
    print(f'\n✓ Sonuçlar kaydedildi: {RESULTS_DIR}/A_nbs_robustness.json')
else:
    print('Deney A skip edildi (RUN_A_NBS_ROBUSTNESS=False)')

DENEY A — NBS k≥3 Robustness Test

--- Warsaw Pact 1980 ---
Üye            Orig NBS   Filt NBS        Δ
--------------------------------------------
GDR             +1.0083    +1.0126  +0.0043
POL             +1.0217    +1.0260  +0.0043
HUN             +0.9833    +0.9876  +0.0043
CZE             +0.9871    +0.9914  +0.0043
ALB             -4.3314    -4.3514  -0.0200
BUL             +0.9840    +0.9865  +0.0025
ROM             +0.9783    +0.9764  -0.0019
RUS             +1.4196    +1.4219  +0.0023

--- Warsaw Pact 1989 ---
Üye            Orig NBS   Filt NBS        Δ
--------------------------------------------
GDR             +1.3596    +1.3664  +0.0067
POL             +1.3482    +1.3549  +0.0067
HUN             -1.3658    -1.3591  +0.0067
CZE             +1.3234    +1.3302  +0.0067
ALB             -4.1806    -4.1869  -0.0062
BUL             +1.3818    +1.3886  +0.0067
ROM             +1.2883    +1.2951  +0.0067
RUS             +1.7718    +1.7376  -0.0343

--- BRICS 2014 ---
Üye         

### A — Paylaşılabilir özet (Claude'a kopyala-yapıştır)

In [ ]:
if RUN_A_NBS_ROBUSTNESS:
    print('=' * 72)
    print('PAYLASIM ÖZETİ — DENEY A')
    print('=' * 72)
    print()
    print('NBS k≥3 filtered vs unfiltered karşılaştırması (6 koalisyon, logit ölçeği):')
    print()
    for name, res in results_A.items():
        print(f'### {name} (v(N) = {res["v_N"]:+.4f})')
        print(f'  Max |Δ|: {res["max_abs_delta"]:.4f}')
        # En büyük 3 değişim
        deltas = []
        for m_str, orig in res['orig_allocation'].items():
            filt = res['filtered_allocation'][m_str]
            deltas.append((m_str, orig, filt, filt - orig))
        deltas.sort(key=lambda x: -abs(x[3]))
        print(f'  En büyük 3 değişim (üye, orig, filtered, Δ):')
        for m_str, o, f, d in deltas[:3]:
            mname = cow_to_name.get(int(m_str), m_str)
            print(f'    {mname:<14}: {o:+.4f} → {f:+.4f}  (Δ={d:+.4f})')
        print()

    # Hungary 1989 odak
    if 'Warsaw Pact 1989' in results_A:
        res = results_A['Warsaw Pact 1989']
        hun_orig = res['orig_allocation']['310']
        hun_filt = res['filtered_allocation']['310']
        print(f'★ HUNGARY 1989 odak: orig NBS = {hun_orig:+.4f}, filtered = {hun_filt:+.4f}')
        print(f'  Paper iddiası: NBS+0.62; |Δ| = {abs(hun_filt - hun_orig):.4f}')


PAYLASIM ÖZETİ — DENEY A

NBS k≥3 filtered vs unfiltered karşılaştırması (6 koalisyon, logit ölçeği):

### Warsaw Pact 1980 (v(N) = +3.0508)
  Max |Δ|: 0.0200
  En büyük 3 değişim (üye, orig, filtered, Δ):
    ALB           : -4.3314 → -4.3514  (Δ=-0.0200)
    CZE           : +0.9871 → +0.9914  (Δ=+0.0043)
    GDR           : +1.0083 → +1.0126  (Δ=+0.0043)

### Warsaw Pact 1989 (v(N) = +2.9267)
  Max |Δ|: 0.0343
  En büyük 3 değişim (üye, orig, filtered, Δ):
    RUS           : +1.7718 → +1.7376  (Δ=-0.0343)
    POL           : +1.3482 → +1.3549  (Δ=+0.0067)
    CZE           : +1.3234 → +1.3302  (Δ=+0.0067)

### BRICS 2014 (v(N) = +3.8095)
  Max |Δ|: 0.0000
  En büyük 3 değişim (üye, orig, filtered, Δ):
    IND           : +0.8090 → +0.8090  (Δ=-0.0000)
    BRA           : +0.7014 → +0.7015  (Δ=+0.0000)
    RUS           : +0.6113 → +0.6113  (Δ=+0.0000)

### ASEAN-10 2010 (v(N) = +2.5854)
  Max |Δ|: 1.0970
  En büyük 3 değişim (üye, orig, filtered, Δ):
    LAO           : -1.5467 → -0

---# Deney B: Shapley Bootstrap Confidence Intervals**Amaç:** Section 5.3, 5.4'teki Shapley değerlerine 95% CI eklemek.**Yöntem:** 10 farklı Monte Carlo seed ile Shapley hesapla, median ± [p2.5, p97.5] aralığı.**Beklenti:** SE < 0.05 (paper'da iddia edilen). Hungary 1989 negative Shapley'in sign-stability'sini kontrol et.

In [ ]:
def monte_carlo_shapley(model, members, year, n_samples=150, use_logit=True, seed=42):
    """Monte Carlo Shapley over random permutations."""
    rng = np.random.default_rng(seed)
    n = len(members)
    shapley = {m: 0.0 for m in members}

    for _ in range(n_samples):
        perm = list(rng.permutation(members))
        for i, m in enumerate(perm):
            S_before = perm[:i]
            S_after  = perm[:i+1]
            v_before = compute_v(model, S_before, year, return_logit=use_logit) if len(S_before) >= 2 else 0.0
            v_after  = compute_v(model, S_after,  year, return_logit=use_logit) if len(S_after)  >= 2 else 0.0
            if v_before is None: v_before = 0.0
            if v_after  is None: v_after  = 0.0
            shapley[m] += (v_after - v_before)

    return {m: shapley[m] / n_samples for m in members}


if RUN_B_SHAPLEY_CI:
    print('=' * 72)
    print('DENEY B — Shapley Bootstrap CI')
    print('=' * 72)

    results_B = {}
    for name, members, year in coalitions_paper:
        print(f'\n--- {name} ---')
        all_shapleys = []
        t0 = time.time()
        for s in tqdm(B_BOOTSTRAP_SEEDS, desc=f'{name}'):
            n_samp = 2000 if len(members) <= 8 else 1500
            sh = monte_carlo_shapley(model, members, year, n_samples=150, seed=s)
            all_shapleys.append(sh)
        print(f'  ({time.time()-t0:.1f}s)')

        # Üye bazlı CI
        summary = {}
        print(f'{"Üye":<12} {"Median":>10} {"SE":>8} {"95% CI":>20}  Sign-stable?')
        for m in members:
            vals = np.array([sh[m] for sh in all_shapleys])
            median = np.median(vals)
            se = np.std(vals, ddof=1)
            ci_lo, ci_hi = np.percentile(vals, [2.5, 97.5])
            sign_stable = (ci_lo > 0) or (ci_hi < 0)  # sign across all bootstraps
            mname = cow_to_name.get(m, str(m))[:11]
            print(f'{mname:<12} {median:>+10.4f} {se:>8.4f}  [{ci_lo:+.3f}, {ci_hi:+.3f}]  {"✓" if sign_stable else "!"}')
            summary[str(m)] = {
                'median': float(median),
                'se': float(se),
                'ci_lo': float(ci_lo),
                'ci_hi': float(ci_hi),
                'sign_stable': bool(sign_stable),
            }

        results_B[name] = {
            'members': members,
            'year': year,
            'n_seeds': len(B_BOOTSTRAP_SEEDS),
            'summary': summary,
        }

    with open(os.path.join(RESULTS_DIR, 'B_shapley_ci.json'), 'w') as f:
        json_mod.dump(results_B, f, indent=2)
    print(f'\n✓ Sonuçlar kaydedildi: {RESULTS_DIR}/B_shapley_ci.json')
else:
    print('Deney B skip edildi (RUN_B_SHAPLEY_CI=False)')

DENEY B — Shapley Bootstrap CI

--- Warsaw Pact 1980 ---


Warsaw Pact 1980:   0%|          | 0/10 [00:00<?, ?it/s]

  (106.9s)
Üye              Median       SE               95% CI  Sign-stable?
GDR             +0.4442   0.0818  [+0.331, +0.594]  ✓
POL             +0.6068   0.0894  [+0.533, +0.777]  ✓
HUN             +0.4826   0.0895  [+0.378, +0.650]  ✓
CZE             +0.5038   0.0966  [+0.412, +0.670]  ✓
ALB             -2.0064   0.1425  [-2.181, -1.766]  ✓
BUL             +0.4409   0.0940  [+0.314, +0.595]  ✓
ROM             +0.5276   0.0837  [+0.426, +0.692]  ✓
RUS             +1.9881   0.1430  [+1.720, +2.099]  ✓

--- Warsaw Pact 1989 ---


Warsaw Pact 1989:   0%|          | 0/10 [00:00<?, ?it/s]

  (107.1s)
Üye              Median       SE               95% CI  Sign-stable?
GDR             +0.5675   0.1051  [+0.471, +0.752]  ✓
POL             +0.9218   0.0680  [+0.825, +1.020]  ✓
HUN             -0.3242   0.0633  [-0.434, -0.237]  ✓
CZE             +0.5575   0.1132  [+0.355, +0.719]  ✓
ALB             -2.1968   0.1819  [-2.452, -1.899]  ✓
BUL             +0.4698   0.0707  [+0.335, +0.542]  ✓
ROM             +0.6700   0.0963  [+0.519, +0.808]  ✓
RUS             +2.2798   0.1546  [+1.967, +2.447]  ✓

--- BRICS 2014 ---


...

  (140.1s)
Üye              Median       SE               95% CI  Sign-stable?
MYA             +0.2298   0.0869  [+0.188, +0.441]  ✓
THI             +0.7366   0.1114  [+0.581, +0.933]  ✓
CAM             -0.1170   0.0495  [-0.157, -0.012]  ✓
LAO             -0.3750   0.0681  [-0.445, -0.271]  ✓
DRV             +0.8041   0.1029  [+0.627, +0.927]  ✓
MAL             +0.4989   0.0754  [+0.434, +0.630]  ✓
SIN             -0.5720   0.1304  [-0.790, -0.394]  ✓
BRU             -0.1512   0.0676  [-0.285, -0.092]  ✓
PHI             +0.2170   0.0618  [+0.098, +0.299]  ✓
INS             +1.2569   0.1147  [+1.098, +1.470]  ✓

--- NATO-26 2010 ---


NATO-26 2010:   0%|          | 0/10 [00:00<?, ?it/s]

  (403.7s)
Üye              Median       SE               95% CI  Sign-stable?
USA             +0.8345   0.0432  [+0.765, +0.890]  ✓
CAN             -0.1140   0.0581  [-0.208, -0.026]  ✓
UKG             +0.2269   0.0330  [+0.182, +0.279]  ✓
NTH             +0.1631   0.0428  [+0.116, +0.246]  ✓
BEL             +0.1584   0.0530  [+0.096, +0.249]  ✓
LUX             -0.0313   0.0412  [-0.123, +0.008]  !
FRN             +0.2509   0.0264  [+0.210, +0.287]  ✓
SPN             +0.2996   0.0525  [+0.209, +0.346]  ✓
POR             +0.1944   0.0492  [+0.141, +0.283]  ✓
GMY             +0.2745   0.0435  [+0.187, +0.320]  ✓
POL             +0.2619   0.0668  [+0.163, +0.363]  ✓
HUN             +0.2017   0.0469  [+0.171, +0.290]  ✓
CZR             +0.1654   0.0386  [+0.143, +0.240]  ✓
SLO             +0.1447   0.0646  [+0.048, +0.245]  ✓
ITA             +0.2304   0.0560  [+0.162, +0.320]  ✓
SLV             -0.1432   0.0482  [-0.213, -0.073]  ✓
GRC             -0.1218   0.0278  [-0.159, -0.076]  ✓
BUL

EU-27 2010:   0%|          | 0/10 [00:00<?, ?it/s]

  (421.6s)
Üye              Median       SE               95% CI  Sign-stable?
UKG             +0.3052   0.0579  [+0.228, +0.400]  ✓
IRE             +0.1094   0.0346  [+0.074, +0.165]  ✓
NTH             +0.1766   0.0352  [+0.136, +0.242]  ✓
BEL             +0.1741   0.0423  [+0.101, +0.217]  ✓
LUX             -0.2003   0.0310  [-0.260, -0.167]  ✓
FRN             +0.3254   0.0612  [+0.272, +0.454]  ✓
SPN             +0.2979   0.0480  [+0.249, +0.393]  ✓
POR             +0.1969   0.0523  [+0.118, +0.272]  ✓
GMY             +0.3508   0.0566  [+0.291, +0.468]  ✓
POL             +0.2645   0.0373  [+0.217, +0.312]  ✓
AUS             +0.1603   0.0753  [+0.090, +0.316]  ✓
HUN             +0.2225   0.0463  [+0.178, +0.317]  ✓
CZR             +0.2298   0.0338  [+0.166, +0.270]  ✓
SLO             +0.1465   0.0463  [+0.098, +0.234]  ✓
ITA             +0.2807   0.0346  [+0.236, +0.336]  ✓
MLT             -0.1943   0.0390  [-0.245, -0.144]  ✓
SLV             +0.0328   0.0319  [-0.017, +0.069]  !
GRC

### B — Paylaşılabilir özet

In [ ]:
if RUN_B_SHAPLEY_CI:
    print('=' * 72)
    print('PAYLASIM ÖZETİ — DENEY B')
    print('=' * 72)
    print()
    print('Shapley 10-seed bootstrap CI özeti (6 koalisyon, logit ölçeği):')
    print()
    for name, res in results_B.items():
        print(f'### {name}')
        # Genel SE istatistiği
        ses = [v['se'] for v in res['summary'].values()]
        print(f'  Min SE: {min(ses):.4f} | Median SE: {np.median(ses):.4f} | Max SE: {max(ses):.4f}')
        n_unstable = sum(1 for v in res['summary'].values() if not v['sign_stable'])
        print(f'  Sign-stable üye sayısı: {len(res["summary"]) - n_unstable}/{len(res["summary"])}')

        # Kritik üyelerin durumu
        critical = []
        if 'Warsaw Pact' in name:
            critical = [('310', 'Hungary'), ('339', 'Albania'), ('365', 'USSR')]
        elif name == 'BRICS 2014':
            critical = [('710', 'China'), ('365', 'Russia')]
        elif name == 'NATO-26 2010':
            critical = [('2', 'USA'), ('310', 'Hungary')]
        elif name == 'ASEAN-10 2010':
            critical = [('850', 'Indonesia'), ('830', 'Singapore')]

        for m_str, label in critical:
            if m_str in res['summary']:
                v = res['summary'][m_str]
                stable = '✓' if v['sign_stable'] else '!'
                print(f'  ★ {label}: {v["median"]:+.4f} ± {v["se"]:.4f}  CI=[{v["ci_lo"]:+.3f}, {v["ci_hi"]:+.3f}]  {stable}')
        print()


PAYLASIM ÖZETİ — DENEY B

Shapley 10-seed bootstrap CI özeti (6 koalisyon, logit ölçeği):

### Warsaw Pact 1980
  Min SE: 0.0818 | Median SE: 0.0918 | Max SE: 0.1430
  Sign-stable üye sayısı: 8/8
  ★ Hungary: +0.4826 ± 0.0895  CI=[+0.378, +0.650]  ✓
  ★ Albania: -2.0064 ± 0.1425  CI=[-2.181, -1.766]  ✓
  ★ USSR: +1.9881 ± 0.1430  CI=[+1.720, +2.099]  ✓

### Warsaw Pact 1989
  Min SE: 0.0633 | Median SE: 0.1007 | Max SE: 0.1819
  Sign-stable üye sayısı: 8/8
  ★ Hungary: -0.3242 ± 0.0633  CI=[-0.434, -0.237]  ✓
  ★ Albania: -2.1968 ± 0.1819  CI=[-2.452, -1.899]  ✓
  ★ USSR: +2.2798 ± 0.1546  CI=[+1.967, +2.447]  ✓

### BRICS 2014
  Min SE: 0.0777 | Median SE: 0.0987 | Max SE: 0.1386
  Sign-stable üye sayısı: 5/5
  ★ China: +1.0399 ± 0.1058  CI=[+0.843, +1.122]  ✓
  ★ Russia: +0.5846 ± 0.1386  CI=[+0.361, +0.771]  ✓

### ASEAN-10 2010
  Min SE: 0.0495 | Median SE: 0.0812 | Max SE: 0.1304
  Sign-stable üye sayısı: 10/10
  ★ Indonesia: +1.2569 ± 0.1147  CI=[+1.098, +1.470]  ✓
  ★ Singapore:

---# Deney C: Multi-Task Ablation (λ_dur = λ_coh = 0)**Amaç:** Section 3.5 ve 6.2'deki ablation iddialarını gerçek sayılarla kapatmak.**Yöntem:** Aynı hiperparametrelerle, ama λ_dur=λ_coh=0 ile retrain → tüm Section 5 metrikleri hesapla.**Beklenen sonuç:** ε* ve Shapley sıralamaları benzer kalırsa multi-task "interpretive grounding" iddiası güçlenir. Eğer çok değişirse paper'ın iddiası daralır.**Süre:** ~20-30 dk (training + post-hoc analysis).

In [ ]:
RUN_C_MULTITASK_ABL = True
if RUN_C_MULTITASK_ABL:
    print('=' * 72)
    print('DENEY C — Multi-Task Ablation')
    print('=' * 72)

    # ----- DATA LOADING -----
    pos_df = pd.read_parquet(os.path.join(COAL_DIR, 'positive_samples.parquet'))
    neg_df = pd.read_parquet(os.path.join(COAL_DIR, 'negative_samples.parquet'))

    def parse_members(s):
        if isinstance(s, list): return [int(x) for x in s]
        if isinstance(s, str):  return [int(x) for x in s.split(',')]
        return []

    def build_samples(df, label):
        samples = []
        for _, row in df.iterrows():
            year = int(row['year'])
            if year not in snapshots: continue
            members = parse_members(row['members_str'])
            c2i = snapshots[year]['code_to_idx']
            valid = [m for m in members if m in c2i]
            if len(valid) < 2: continue
            samples.append({
                'year': year,
                'members': valid,
                'member_idx': [c2i[m] for m in valid],
                'label': label,
                'duration': float(row.get('total_duration', 0)) if label == 1 else 0.0,
                'cohesion': float(row.get('internal_vote_agreement', 0) or 0) if label == 1 else 0.0,
            })
        return samples

    all_samples = build_samples(pos_df, 1) + build_samples(neg_df, 0)
    test_samples = [s for s in all_samples if s['year'] > VAL_END]
    pre_test = [s for s in all_samples if s['year'] <= VAL_END]

    SEED_C = 42
    rng_c = np.random.default_rng(SEED_C)
    idx = np.arange(len(pre_test))
    rng_c.shuffle(idx)
    val_size = int(0.15 * len(pre_test))
    val_set = set(idx[:val_size].tolist())
    train_samples = [pre_test[i] for i in range(len(pre_test)) if i not in val_set]
    val_samples   = [pre_test[i] for i in range(len(pre_test)) if i in val_set]
    print(f'Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}')

    # ----- TRAIN: SINGLE-TASK (λ=0) -----
    torch.manual_seed(SEED_C); np.random.seed(SEED_C); random.seed(SEED_C)
    model_st = FullRichGNN(NUM_FEATURES).to(DEVICE)

    EPOCHS = 20; LR = 1e-3; ENCODER_LR = 1e-4; WEIGHT_DECAY = 1e-3
    LABEL_SMOOTHING = 0.1; PATIENCE = 25; POS_WEIGHT = 2.5; BATCH_SIZE = 32
    LAMBDA_DUR = 0.0  # << ABLATION
    LAMBDA_COH = 0.0  # << ABLATION

    optimizer = torch.optim.Adam([
        {'params': model_st.encoder.parameters(), 'lr': ENCODER_LR},
        {'params': model_st.head.parameters(),    'lr': LR},
    ], weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    def bce_smooth(logit, target, smoothing=0.1):
        target_s = target * (1 - smoothing) + 0.5 * smoothing
        w = torch.tensor(POS_WEIGHT if target.item() > 0.5 else 1.0, device=DEVICE)
        return F.binary_cross_entropy_with_logits(logit, target_s, weight=w)

    def evaluate(m, samples):
        m.eval()
        logits, labels = [], []
        with torch.no_grad():
            for s in samples:
                v, _, _ = m.forward_sample(snapshots[s['year']], s['member_idx'])
                logits.append(v.item()); labels.append(s['label'])
        logits, labels = np.array(logits), np.array(labels)
        probs = 1/(1+np.exp(-logits))
        p, r, _ = precision_recall_curve(labels, probs)
        f1 = (2*p*r/(p+r+1e-10)).max()
        return {'AUC': roc_auc_score(labels, probs),
                'PR-AUC': average_precision_score(labels, probs),
                'F1_opt': f1}

    best_val_auc, best_state, pat = 0, None, 0
    print('\nSingle-task training başlıyor (λ_dur=λ_coh=0)...')
    t0 = time.time()
    for epoch in range(EPOCHS):
        model_st.train()
        random.shuffle(train_samples)
        epoch_loss, nb = 0.0, 0
        for i in range(0, len(train_samples), BATCH_SIZE):
            batch = train_samples[i:i+BATCH_SIZE]
            optimizer.zero_grad()
            bl, nv = 0.0, 0
            for s in batch:
                validity, dur, coh = model_st.forward_sample(snapshots[s['year']], s['member_idx'])
                target = torch.tensor(float(s['label']), device=DEVICE)
                loss = bce_smooth(validity, target, LABEL_SMOOTHING)
                # NO multi-task loss (ablation)
                bl = bl + loss; nv += 1
            if nv == 0: continue
            bl = bl / nv
            bl.backward()
            torch.nn.utils.clip_grad_norm_(model_st.parameters(), 1.0)
            optimizer.step()
            epoch_loss += bl.item(); nb += 1
        scheduler.step()
        train_loss = epoch_loss / max(nb, 1)
        vm = evaluate(model_st, val_samples)
        if vm['AUC'] > best_val_auc:
            best_val_auc = vm['AUC']
            best_state = {k: v.cpu().clone() for k, v in model_st.state_dict().items()}
            pat = 0; flag = ' ←'
        else:
            pat += 1; flag = ''
        if epoch % 10 == 0 or flag:
            print(f'  Ep {epoch:3d} | loss={train_loss:.4f} | val AUC={vm["AUC"]:.4f}{flag}')
        if pat >= PATIENCE:
            print(f'  Early stop ep{epoch}'); break

    model_st.load_state_dict(best_state)
    torch.save(best_state, os.path.join(MODELS_DIR, 'full_rich_gnn_singletask.pt'))
    test_metrics_st = evaluate(model_st, test_samples)
    print(f'\n✓ Single-task training tamamlandı ({time.time()-t0:.1f}s)')
    print(f'  Test AUC={test_metrics_st["AUC"]:.4f}, F1={test_metrics_st["F1_opt"]:.4f}')

    # Multi-task baseline metrics (paper: AUC 0.938, F1 0.822)
    test_metrics_mt = evaluate(model, test_samples)
    print(f'\nMulti-task baseline:    AUC={test_metrics_mt["AUC"]:.4f}, F1={test_metrics_mt["F1_opt"]:.4f}')
    print(f'Single-task ablation:    AUC={test_metrics_st["AUC"]:.4f}, F1={test_metrics_st["F1_opt"]:.4f}')

    # ----- POST-HOC ANALYSIS on coalitions -----
    print('\n--- Section 5 metrics: multi-task vs single-task ---')
    results_C = {
        'test_metrics_multitask':  test_metrics_mt,
        'test_metrics_singletask': test_metrics_st,
        'coalitions': {},
    }

    def least_core_quick(m_model, members, year, max_coalitions=800):
        n = len(members)
        v_N = compute_v(m_model, members, year, return_logit=True)
        if v_N is None: return None
        coalition_specs = []
        if 2**n <= max_coalitions + n + 2:
            for k in range(2, n):
                for combo in combinations(range(n), k):
                    sub_members = [members[i] for i in combo]
                    v_S = compute_v(m_model, sub_members, year, return_logit=True)
                    if v_S is not None:
                        indicator = [1.0 if i in combo else 0.0 for i in range(n)]
                        coalition_specs.append((indicator, v_S))
        else:
            rng = np.random.default_rng(42)
            seen = set(); attempts = 0
            while len(coalition_specs) < max_coalitions and attempts < max_coalitions * 3:
                attempts += 1
                k = int(rng.integers(2, n))
                combo = tuple(sorted(rng.choice(n, k, replace=False).tolist()))
                if combo in seen: continue
                seen.add(combo)
                sub_members = [members[i] for i in combo]
                v_S = compute_v(m_model, sub_members, year, return_logit=True)
                if v_S is not None:
                    indicator = [1.0 if i in combo else 0.0 for i in range(n)]
                    coalition_specs.append((indicator, v_S))
        if not coalition_specs: return None
        c = [0.0]*n + [1.0]
        A_ub, b_ub = [], []
        n_blocking = 0
        for indicator, v_S in coalition_specs:
            A_ub.append([-x for x in indicator] + [-1.0])
            b_ub.append(-v_S)
            if v_S > v_N: n_blocking += 1
        A_eq = [[1.0]*n + [0.0]]; b_eq = [v_N]
        bounds = [(None, None)] * (n + 1)
        res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
        if not res.success: return None
        return {
            'epsilon': float(res.x[-1]),
            'v_N': float(v_N),
            'n_coalitions': len(coalition_specs),
            'pct_blocking': 100 * n_blocking / len(coalition_specs),
        }

    for name, members, year in coalitions_paper:
        print(f'\n  {name}')
        lc_mt = least_core_quick(model,    members, year)
        lc_st = least_core_quick(model_st, members, year)
        sh_mt = monte_carlo_shapley(model,    members, year, n_samples=150, seed=42)
        sh_st = monte_carlo_shapley(model_st, members, year, n_samples=150, seed=42)

        # Top-3 Shapley karşılaştırma
        top3_mt = sorted(sh_mt.items(), key=lambda x: -x[1])[:3]
        top3_st = sorted(sh_st.items(), key=lambda x: -x[1])[:3]

        print(f'    v(N):  MT={lc_mt["v_N"]:+.3f}  ST={lc_st["v_N"]:+.3f}')
        print(f'    ε*:    MT={lc_mt["epsilon"]:+.3f}  ST={lc_st["epsilon"]:+.3f}')
        print(f'    %blk:  MT={lc_mt["pct_blocking"]:.1f}%   ST={lc_st["pct_blocking"]:.1f}%')
        print(f'    Top3 MT: {[(cow_to_name.get(m, m), f"{v:+.2f}") for m,v in top3_mt]}')
        print(f'    Top3 ST: {[(cow_to_name.get(m, m), f"{v:+.2f}") for m,v in top3_st]}')

        results_C['coalitions'][name] = {
            'members': members, 'year': year,
            'multitask':  {'v_N': lc_mt['v_N'], 'epsilon': lc_mt['epsilon'], 'pct_blocking': lc_mt['pct_blocking'],
                           'shapley': {str(m): float(v) for m,v in sh_mt.items()}},
            'singletask': {'v_N': lc_st['v_N'], 'epsilon': lc_st['epsilon'], 'pct_blocking': lc_st['pct_blocking'],
                           'shapley': {str(m): float(v) for m,v in sh_st.items()}},
        }

    with open(os.path.join(RESULTS_DIR, 'C_multitask_ablation.json'), 'w') as f:
        json_mod.dump(results_C, f, indent=2)
    print(f'\n✓ Sonuçlar kaydedildi: {RESULTS_DIR}/C_multitask_ablation.json')
else:
    print('Deney C skip edildi (RUN_C_MULTITASK_ABL=False)')

DENEY C — Multi-Task Ablation
Train: 2248 | Val: 396 | Test: 358

Single-task training başlıyor (λ_dur=λ_coh=0)...
  Ep   0 | loss=0.8738 | val AUC=0.8288 ←
  Ep   1 | loss=0.7220 | val AUC=0.9210 ←
  Ep   2 | loss=0.6368 | val AUC=0.9529 ←
  Ep   3 | loss=0.5877 | val AUC=0.9642 ←
  Ep   4 | loss=0.5909 | val AUC=0.9713 ←
  Ep   5 | loss=0.5190 | val AUC=0.9729 ←
  Ep   7 | loss=0.5037 | val AUC=0.9769 ←
  Ep   8 | loss=0.4913 | val AUC=0.9803 ←
  Ep   9 | loss=0.4678 | val AUC=0.9821 ←
  Ep  10 | loss=0.4769 | val AUC=0.9800
  Ep  11 | loss=0.4531 | val AUC=0.9833 ←
  Ep  12 | loss=0.4533 | val AUC=0.9847 ←
  Ep  13 | loss=0.4482 | val AUC=0.9859 ←
  Ep  15 | loss=0.4273 | val AUC=0.9864 ←
  Ep  16 | loss=0.4281 | val AUC=0.9878 ←
  Ep  17 | loss=0.4161 | val AUC=0.9879 ←

✓ Single-task training tamamlandı (470.2s)
  Test AUC=0.9260, F1=0.7938

Multi-task baseline:    AUC=0.9384, F1=0.8218
Single-task ablation:    AUC=0.9260, F1=0.7938

--- Section 5 metrics: multi-task vs single-tas

### C — Paylaşılabilir özet

In [ ]:
if RUN_C_MULTITASK_ABL:
    print('=' * 72)
    print('PAYLASIM ÖZETİ — DENEY C')
    print('=' * 72)
    print()
    print(f'Test performance:  MT AUC={results_C["test_metrics_multitask"]["AUC"]:.4f}  ST AUC={results_C["test_metrics_singletask"]["AUC"]:.4f}')
    print()
    print(f'{"Koalisyon":<22} {"ε* MT":>8} {"ε* ST":>8} {"Δε*":>8}  {"%blk MT":>8} {"%blk ST":>8}')
    print('-' * 76)
    for name, res in results_C['coalitions'].items():
        mt = res['multitask']; st = res['singletask']
        d_eps = st['epsilon'] - mt['epsilon']
        print(f'{name:<22} {mt["epsilon"]:>+8.3f} {st["epsilon"]:>+8.3f} {d_eps:>+8.3f}  '
              f'{mt["pct_blocking"]:>7.1f}% {st["pct_blocking"]:>7.1f}%')

    # Hungary 1989 odak
    if 'Warsaw Pact 1989' in results_C['coalitions']:
        warsaw89 = results_C['coalitions']['Warsaw Pact 1989']
        hun_mt = warsaw89['multitask']['shapley'].get('310', 0)
        hun_st = warsaw89['singletask']['shapley'].get('310', 0)
        print(f'\n★ Hungary 1989 Shapley: MT={hun_mt:+.4f}, ST={hun_st:+.4f}, Δ={hun_st-hun_mt:+.4f}')
        print(f'  Sign-stable: {(hun_mt < 0) == (hun_st < 0)}')


PAYLASIM ÖZETİ — DENEY C

Test performance:  MT AUC=0.9384  ST AUC=0.9260

Koalisyon                 ε* MT    ε* ST      Δε*   %blk MT  %blk ST
----------------------------------------------------------------------------
Warsaw Pact 1980         +1.837   +2.935   +1.098     25.6%    48.8%
Warsaw Pact 1989         +1.619   +3.543   +1.923     37.0%    50.0%
BRICS 2014               +2.156   +3.299   +1.143     36.0%    36.0%
ASEAN-10 2010            +2.269   +2.839   +0.569     53.0%    42.8%
NATO-26 2010             +2.606   +3.492   +0.886     25.2%    29.4%
EU-27 2010               +2.547   +3.360   +0.813     30.5%    33.9%

★ Hungary 1989 Shapley: MT=-0.3801, ST=-0.0828, Δ=+0.2973
  Sign-stable: True


---# Deney D: Multi-Seed Retrain (5 seed)**Amaç:** Section 5 metriklerine seed-varyasyonu CI eklemek.**Yöntem:** Aynı multi-task setup ile 5 farklı seed retrain, her seed için ε* ve Shapley'leri kaydet, median±range raporla.**Süre:** ~5-7 dk × 5 = 30-40 dk training + 20 dk analysis. Toplam ~60 dk.

In [ ]:
RUN_D_MULTISEED = True
if RUN_D_MULTISEED:
    print('=' * 72)
    print(f'DENEY D — Multi-Seed Retrain (n={MULTISEED_N})')
    print('=' * 72)

    # Data loading (C ile aynı, eğer C koşmadıysa burada yükle)
    if 'train_samples' not in globals():
        pos_df = pd.read_parquet(os.path.join(COAL_DIR, 'positive_samples.parquet'))
        neg_df = pd.read_parquet(os.path.join(COAL_DIR, 'negative_samples.parquet'))
        def parse_members(s):
            if isinstance(s, list): return [int(x) for x in s]
            if isinstance(s, str):  return [int(x) for x in s.split(',')]
            return []
        def build_samples(df, label):
            samples = []
            for _, row in df.iterrows():
                year = int(row['year'])
                if year not in snapshots: continue
                members = parse_members(row['members_str'])
                c2i = snapshots[year]['code_to_idx']
                valid = [m for m in members if m in c2i]
                if len(valid) < 2: continue
                samples.append({'year': year, 'members': valid,
                                'member_idx': [c2i[m] for m in valid], 'label': label,
                                'duration': float(row.get('total_duration', 0)) if label == 1 else 0.0,
                                'cohesion': float(row.get('internal_vote_agreement', 0) or 0) if label == 1 else 0.0,})
            return samples
        all_samples = build_samples(pos_df, 1) + build_samples(neg_df, 0)
        test_samples = [s for s in all_samples if s['year'] > VAL_END]
        pre_test = [s for s in all_samples if s['year'] <= VAL_END]

    def train_one_seed(seed_value):
        torch.manual_seed(seed_value); np.random.seed(seed_value); random.seed(seed_value)
        rng_d = np.random.default_rng(seed_value)
        idx = np.arange(len(pre_test))
        rng_d.shuffle(idx)
        val_size = int(0.15 * len(pre_test))
        val_set = set(idx[:val_size].tolist())
        train_s = [pre_test[i] for i in range(len(pre_test)) if i not in val_set]
        val_s   = [pre_test[i] for i in range(len(pre_test)) if i in val_set]

        m = FullRichGNN(NUM_FEATURES).to(DEVICE)
        EPOCHS = 20; PATIENCE = 25
        opt = torch.optim.Adam([
            {'params': m.encoder.parameters(), 'lr': 1e-4},
            {'params': m.head.parameters(),    'lr': 1e-3},
        ], weight_decay=1e-3)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
        def bce_s(l, t, sm=0.1):
            ts = t * (1 - sm) + 0.5 * sm
            w = torch.tensor(2.5 if t.item() > 0.5 else 1.0, device=DEVICE)
            return F.binary_cross_entropy_with_logits(l, ts, weight=w)
        def eval_m(mm, samples):
            mm.eval()
            logits, labels = [], []
            with torch.no_grad():
                for s in samples:
                    v, _, _ = mm.forward_sample(snapshots[s['year']], s['member_idx'])
                    logits.append(v.item()); labels.append(s['label'])
            logits, labels = np.array(logits), np.array(labels)
            probs = 1/(1+np.exp(-logits))
            p, r, _ = precision_recall_curve(labels, probs)
            f1 = (2*p*r/(p+r+1e-10)).max()
            return {'AUC': roc_auc_score(labels, probs),
                    'F1': f1}

        best_auc, best_state, pat = 0, None, 0
        for epoch in range(EPOCHS):
            m.train(); random.shuffle(train_s)
            for i in range(0, len(train_s), 32):
                batch = train_s[i:i+32]
                opt.zero_grad()
                bl, nv = 0.0, 0
                for s in batch:
                    v, dur, coh = m.forward_sample(snapshots[s['year']], s['member_idx'])
                    tg = torch.tensor(float(s['label']), device=DEVICE)
                    loss = bce_s(v, tg)
                    if s['label'] == 1:
                        dur_t = torch.log1p(torch.tensor(s['duration'], device=DEVICE))
                        loss = loss + 0.3 * F.mse_loss(dur, dur_t)
                        if s['cohesion'] > 0:
                            coh_t = torch.tensor(s['cohesion'], device=DEVICE)
                            loss = loss + 0.3 * F.mse_loss(coh, coh_t)
                    bl = bl + loss; nv += 1
                if nv == 0: continue
                bl = bl / nv
                bl.backward()
                torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                opt.step()
            sched.step()
            vm = eval_m(m, val_s)
            if vm['AUC'] > best_auc:
                best_auc = vm['AUC']
                best_state = {k: vv.cpu().clone() for k, vv in m.state_dict().items()}
                pat = 0
            else:
                pat += 1
            if pat >= PATIENCE: break
        m.load_state_dict(best_state)
        test_m = eval_m(m, test_samples)
        return m, test_m, best_auc

    # Train all seeds
    results_D = {'seeds': MULTISEED_SEEDS, 'per_seed': {}}
    for seed_value in MULTISEED_SEEDS:
        print(f'\n--- Seed {seed_value} ---')
        t0 = time.time()
        m_s, test_m, best_val = train_one_seed(seed_value)
        print(f'  Training {time.time()-t0:.1f}s | val AUC={best_val:.4f} | test AUC={test_m["AUC"]:.4f}')

        # Coalition metrics
        per_coalition = {}
        for name, members, year in coalitions_paper:
            v_N = compute_v(m_s, members, year, return_logit=True)
            sh  = monte_carlo_shapley(m_s, members, year, n_samples=150, seed=42)
            # ε* shortcut: blocking sayısı yeterli
            from itertools import combinations
            n = len(members)
            n_blocking = 0; n_total = 0; max_gap = 0
            if 2**n <= 800 + n + 2:
                for k in range(2, n):
                    for combo in combinations(range(n), k):
                        v_S = compute_v(m_s, [members[i] for i in combo], year, return_logit=True)
                        if v_S is not None:
                            n_total += 1
                            if v_S > v_N:
                                n_blocking += 1
                                max_gap = max(max_gap, v_S - v_N)
            else:
                rng = np.random.default_rng(42); seen=set()
                while n_total < 500:
                    k = int(rng.integers(2, n))
                    combo = tuple(sorted(rng.choice(n, k, replace=False).tolist()))
                    if combo in seen: continue
                    seen.add(combo)
                    v_S = compute_v(m_s, [members[i] for i in combo], year, return_logit=True)
                    if v_S is not None:
                        n_total += 1
                        if v_S > v_N:
                            n_blocking += 1
                            max_gap = max(max_gap, v_S - v_N)
            per_coalition[name] = {
                'v_N': float(v_N) if v_N else None,
                'pct_blocking': 100 * n_blocking / n_total if n_total else 0,
                'max_gap': float(max_gap),
                'shapley': {str(m): float(sh[m]) for m in members},
            }
        results_D['per_seed'][str(seed_value)] = {
            'val_auc': float(best_val),
            'test_metrics': {k: float(v) for k, v in test_m.items()},
            'coalitions': per_coalition,
        }

    with open(os.path.join(RESULTS_DIR, 'D_multiseed.json'), 'w') as f:
        json_mod.dump(results_D, f, indent=2)
    print(f'\n✓ Sonuçlar kaydedildi: {RESULTS_DIR}/D_multiseed.json')
else:
    print('Deney D skip edildi (RUN_D_MULTISEED=False)')

DENEY D — Multi-Seed Retrain (n=5)

--- Seed 42 ---
  Training 482.7s | val AUC=0.9876 | test AUC=0.9242

--- Seed 137 ---
  Training 482.6s | val AUC=0.9801 | test AUC=0.9192

--- Seed 256 ---
  Training 485.6s | val AUC=0.9763 | test AUC=0.9290

--- Seed 511 ---
  Training 482.5s | val AUC=0.9749 | test AUC=0.9227

--- Seed 1024 ---
  Training 481.8s | val AUC=0.9927 | test AUC=0.9103

✓ Sonuçlar kaydedildi: /content/drive/MyDrive/coalition_gnn/geb_experiments/D_multiseed.json


### D — Paylaşılabilir özet

In [ ]:
if RUN_D_MULTISEED:
    print('=' * 72)
    print('PAYLASIM ÖZETİ — DENEY D')
    print('=' * 72)
    print()

    # Test AUC across seeds
    aucs = [results_D['per_seed'][str(s)]['test_metrics']['AUC'] for s in MULTISEED_SEEDS]
    print(f'Test AUC across {len(MULTISEED_SEEDS)} seeds:')
    print(f'  Median: {np.median(aucs):.4f}  Range: [{min(aucs):.4f}, {max(aucs):.4f}]')
    print(f'  Paper baseline (seed 42): 0.938')
    print()

    # Coalition metrics across seeds
    for name, _, _ in coalitions_paper:
        v_Ns = [results_D['per_seed'][str(s)]['coalitions'][name]['v_N'] for s in MULTISEED_SEEDS]
        blockings = [results_D['per_seed'][str(s)]['coalitions'][name]['pct_blocking'] for s in MULTISEED_SEEDS]
        max_gaps = [results_D['per_seed'][str(s)]['coalitions'][name]['max_gap'] for s in MULTISEED_SEEDS]
        print(f'{name}:')
        print(f'  v(N):       median={np.median(v_Ns):+.3f}  range=[{min(v_Ns):+.3f}, {max(v_Ns):+.3f}]')
        print(f'  %blocking:  median={np.median(blockings):.1f}%  range=[{min(blockings):.1f}%, {max(blockings):.1f}%]')
        print(f'  max gap:    median={np.median(max_gaps):+.3f}  range=[{min(max_gaps):+.3f}, {max(max_gaps):+.3f}]')

    # Hungary 1989 stability across seeds
    if 'Warsaw Pact 1989' in results_D['per_seed'][str(MULTISEED_SEEDS[0])]['coalitions']:
        print('\n★ Hungary 1989 Shapley across seeds:')
        hun_vals = []
        for s in MULTISEED_SEEDS:
            sh = results_D['per_seed'][str(s)]['coalitions']['Warsaw Pact 1989']['shapley']
            if '310' in sh:
                hun_vals.append(sh['310'])
        if hun_vals:
            print(f'  Median: {np.median(hun_vals):+.4f}')
            print(f'  Range: [{min(hun_vals):+.4f}, {max(hun_vals):+.4f}]')
            n_neg = sum(1 for v in hun_vals if v < 0)
            print(f'  Negative in {n_neg}/{len(hun_vals)} seeds (sign-stability)')


PAYLASIM ÖZETİ — DENEY D

Test AUC across 5 seeds:
  Median: 0.9227  Range: [0.9103, 0.9290]
  Paper baseline (seed 42): 0.938

Warsaw Pact 1980:
  v(N):       median=+2.519  range=[+1.948, +3.131]
  %blocking:  median=48.8%  range=[48.8%, 48.8%]
  max gap:    median=+2.685  range=[+1.855, +3.547]
Warsaw Pact 1989:
  v(N):       median=+3.090  range=[+2.477, +3.648]
  %blocking:  median=48.8%  range=[48.0%, 52.0%]
  max gap:    median=+2.737  range=[+1.916, +3.760]
BRICS 2014:
  v(N):       median=+3.380  range=[+2.521, +4.323]
  %blocking:  median=32.0%  range=[24.0%, 44.0%]
  max gap:    median=+1.081  range=[+0.959, +1.549]
ASEAN-10 2010:
  v(N):       median=+2.887  range=[+1.762, +3.395]
  %blocking:  median=31.8%  range=[13.6%, 43.2%]
  max gap:    median=+1.813  range=[+0.557, +4.098]
NATO-26 2010:
  v(N):       median=+3.535  range=[+2.895, +3.877]
  %blocking:  median=25.8%  range=[23.4%, 35.2%]
  max gap:    median=+0.975  range=[+0.929, +1.646]
EU-27 2010:
  v(N):       medi

---# Sonuçların paylaşılmasıTüm sonuçlar `{RESULTS_DIR}` klasöründe JSON olarak kayıtlı. Aşağıdaki hücre tek bir özet metin dosyası üretir, Claude'a yapıştırılabilir.

In [ ]:
# Tüm experimentleri tek bir paylaşılabilir özet dosyada birleştir
summary_lines = []
summary_lines.append('=' * 72)
summary_lines.append('GEB PAPER EMPIRIK GÜÇLENDIRME — TOPLU SONUÇ ÖZETİ')
summary_lines.append('=' * 72)
summary_lines.append('')

if RUN_A_NBS_ROBUSTNESS:
    summary_lines.append('[DENEY A — NBS k≥3 robustness] Tamamlandı.')
    for name, res in results_A.items():
        summary_lines.append(f'  {name}: max |Δ| = {res["max_abs_delta"]:.4f}')
    summary_lines.append('')

if RUN_B_SHAPLEY_CI:
    summary_lines.append('[DENEY B — Shapley bootstrap CI] Tamamlandı.')
    for name, res in results_B.items():
        ses = [v['se'] for v in res['summary'].values()]
        n_stable = sum(1 for v in res['summary'].values() if v['sign_stable'])
        summary_lines.append(f'  {name}: median SE={np.median(ses):.4f}, sign-stable {n_stable}/{len(res["summary"])}')
    summary_lines.append('')

if RUN_C_MULTITASK_ABL:
    summary_lines.append('[DENEY C — Multi-task ablation] Tamamlandı.')
    summary_lines.append(f'  MT AUC: {results_C["test_metrics_multitask"]["AUC"]:.4f}')
    summary_lines.append(f'  ST AUC: {results_C["test_metrics_singletask"]["AUC"]:.4f}')
    summary_lines.append('')

if RUN_D_MULTISEED:
    summary_lines.append(f'[DENEY D — Multi-seed retrain] Tamamlandı ({len(MULTISEED_SEEDS)} seed).')
    aucs = [results_D['per_seed'][str(s)]['test_metrics']['AUC'] for s in MULTISEED_SEEDS]
    summary_lines.append(f'  Test AUC median: {np.median(aucs):.4f}, range: [{min(aucs):.4f}, {max(aucs):.4f}]')
    summary_lines.append('')

summary_lines.append('JSON detay dosyaları:')
for f in os.listdir(RESULTS_DIR):
    summary_lines.append(f'  {RESULTS_DIR}/{f}')

summary_text = '\n'.join(summary_lines)
print(summary_text)

# Dosyaya kaydet
with open(os.path.join(RESULTS_DIR, 'SUMMARY.txt'), 'w') as f:
    f.write(summary_text)
print(f'\n✓ Özet: {RESULTS_DIR}/SUMMARY.txt')

GEB PAPER EMPIRIK GÜÇLENDIRME — TOPLU SONUÇ ÖZETİ

[DENEY A — NBS k≥3 robustness] Tamamlandı.
  Warsaw Pact 1980: max |Δ| = 0.0200
  Warsaw Pact 1989: max |Δ| = 0.0343
  BRICS 2014: max |Δ| = 0.0000
  ASEAN-10 2010: max |Δ| = 1.0970
  NATO-26 2010: max |Δ| = 0.6789
  EU-27 2010: max |Δ| = 1.3765

[DENEY B — Shapley bootstrap CI] Tamamlandı.
  Warsaw Pact 1980: median SE=0.0918, sign-stable 8/8
  Warsaw Pact 1989: median SE=0.1007, sign-stable 8/8
  BRICS 2014: median SE=0.0987, sign-stable 5/5
  ASEAN-10 2010: median SE=0.0812, sign-stable 10/10
  NATO-26 2010: median SE=0.0453, sign-stable 23/26
  EU-27 2010: median SE=0.0463, sign-stable 21/27

[DENEY C — Multi-task ablation] Tamamlandı.
  MT AUC: 0.9384
  ST AUC: 0.9260

[DENEY D — Multi-seed retrain] Tamamlandı (5 seed).
  Test AUC median: 0.9227, range: [0.9103, 0.9290]

JSON detay dosyaları:
  /content/drive/MyDrive/coalition_gnn/geb_experiments/A_nbs_robustness.json
  /content/drive/MyDrive/coalition_gnn/geb_experiments/B_shaple